# neo-vs-morpheus: fine-tune Laya on duel decisions (Colab GPU)

Trains Laya `typed-decisions` on duel states labeled by a teacher (Jev), using
`training/finetune.py` from the repo. On a Mac the script can only train the decision head;
on a GPU it trains the whole encoder too (`--mode top --train-top-layers 28`).

1. Runtime → Change runtime type → **T4 GPU** (or better).
2. Run the cells; the upload cell asks for `training/data/labeled.jsonl` from your machine.
3. The last cell downloads `neo-duel.zip`. Unzip it into `training/checkpoints/` and serve it with
   `scripts/laya-native.sh start training/checkpoints/neo-duel`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Switch the runtime to a GPU first (Runtime -> Change runtime type)."

In [ ]:
!pip install -q "laya==0.3.20"
!git clone -q --depth 1 https://github.com/hakkisagdic/neo-vs-morpheus.git
%cd neo-vs-morpheus

In [ ]:
# Upload training/data/labeled.jsonl (made locally by `npm run nvm -- distill label`).
import os
from google.colab import files
os.makedirs("training/data", exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    os.replace(name, "training/data/labeled.jsonl")
!wc -l training/data/labeled.jsonl

In [ ]:
# Full fine-tune: every encoder layer plus the head. Held-out agreement with the teacher is
# printed before training and after each epoch.
!python -u training/finetune.py --mode top --train-top-layers 28 --epochs 3 --batch 8 --accum 2 \
    --data training/data/labeled.jsonl --out training/checkpoints/neo-duel

In [ ]:
!cd training/checkpoints && zip -qr ../../neo-duel.zip neo-duel && ls -lh ../../neo-duel.zip
from google.colab import files
files.download("neo-duel.zip")